# HIT401 Simple Scam Email Project — Part 2

## Goal
Train one understandable baseline model and test whether it works on unfamiliar email sources.

**Model:** TF-IDF + Logistic Regression

- TF-IDF converts words and short phrases into numbers.
- Logistic Regression learns which numeric word patterns are associated with scam and legitimate emails.

In [ ]:
# Cell 1: Import the tools
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

pd.set_option('display.max_colwidth', 120)
print('Tools loaded successfully.')

## Step 1 — Upload the prepared data
Upload `hit401_prepared_data.zip`, which was created by Part 1.

In [ ]:
# Cell 2: Upload and extract the prepared CSV files
zip_path = Path('/content/hit401_prepared_data.zip')
if not zip_path.exists():
    from google.colab import files
    print('Choose hit401_prepared_data.zip from your computer.')
    uploaded = files.upload()
    if 'hit401_prepared_data.zip' not in uploaded:
        raise FileNotFoundError('Please upload hit401_prepared_data.zip.')

data_dir = Path('/content/hit401_part2_data')
with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(data_dir)

training_data = pd.read_csv(data_dir / 'training_data.csv')
external_test_data = pd.read_csv(data_dir / 'external_test_data.csv')

print('Training rows:', len(training_data))
print('External test rows:', len(external_test_data))
display(training_data.head(3))

## Step 2 — Create an internal validation group
We keep 20% of the training-source emails aside. The model learns from the other 80%. This internal validation checks performance on familiar sources. The separate external test checks performance on unfamiliar sources.

In [ ]:
# Cell 3: Split the training-source data
X_develop, X_internal, y_develop, y_internal = train_test_split(
    training_data['text'],
    training_data['label'],
    test_size=0.20,
    random_state=42,
    stratify=training_data['label'],
)

print('Emails used to train the model:', len(X_develop))
print('Emails used for internal validation:', len(X_internal))
print('Emails used for external testing:', len(external_test_data))

## Step 3 — Convert email language into numbers
The model cannot read raw sentences. TF-IDF creates one numeric column for each useful word or two-word phrase. `fit_transform` learns the vocabulary from training emails only. `transform` applies that same vocabulary to validation and test emails.

In [ ]:
# Cell 4: Fit TF-IDF only on the development/training emails
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
)

X_develop_tfidf = vectorizer.fit_transform(X_develop)
X_internal_tfidf = vectorizer.transform(X_internal)
X_external_tfidf = vectorizer.transform(external_test_data['text'])

print('Training matrix shape:', X_develop_tfidf.shape)
print('Number of learned word features:', len(vectorizer.get_feature_names_out()))

## Step 4 — Train Logistic Regression
Training means the model examines the TF-IDF numbers together with the known labels. It learns weights that help separate scam (`1`) from legitimate (`0`) emails.

In [ ]:
# Cell 5: Train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_develop_tfidf, y_develop)
print('Model training completed.')

## Step 5 — Evaluate the model

- **Accuracy:** percentage of all emails classified correctly.
- **Precision:** when the model says scam, how often it is correct.
- **Recall:** percentage of real scams the model catches.
- **F1-score:** one combined measure of precision and recall.

For scam detection, recall matters because low recall means many scams were missed.

In [ ]:
# Cell 6: Make predictions and calculate results
internal_predictions = model.predict(X_internal_tfidf)
external_predictions = model.predict(X_external_tfidf)

def metric_row(test_name, actual, predicted):
    return {
        'test': test_name,
        'accuracy': accuracy_score(actual, predicted),
        'precision_scam': precision_score(actual, predicted, zero_division=0),
        'recall_scam': recall_score(actual, predicted, zero_division=0),
        'f1_scam': f1_score(actual, predicted, zero_division=0),
    }

results = pd.DataFrame([
    metric_row('Internal validation (familiar sources)', y_internal, internal_predictions),
    metric_row('External test (unfamiliar sources)', external_test_data['label'], external_predictions),
]).set_index('test')

results.round(4)

In [ ]:
# Cell 7: Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tests = [
    ('Internal validation', y_internal, internal_predictions),
    ('External test', external_test_data['label'], external_predictions),
]

for axis, (title, actual, predicted) in zip(axes, tests):
    matrix = confusion_matrix(actual, predicted, labels=[0, 1])
    ConfusionMatrixDisplay(matrix, display_labels=['Legitimate', 'Scam']).plot(
        ax=axis, cmap='Blues', colorbar=False
    )
    axis.set_title(title)

plt.tight_layout()
plt.show()

### Reading a confusion matrix
- Top-left: legitimate emails correctly classified.
- Top-right: legitimate emails incorrectly flagged as scams.
- Bottom-left: scams incorrectly classified as legitimate. These are **missed scams**.
- Bottom-right: scams correctly detected.

In [ ]:
# Cell 8: Compare familiar-source and unfamiliar-source performance
results.plot(kind='bar', figsize=(10, 5), ylim=(0, 1), color=['#4C78A8', '#F2CF5B', '#E45756', '#72B7B2'])
plt.title('Logistic Regression performance')
plt.ylabel('Score')
plt.xlabel('')
plt.xticks(rotation=10, ha='right')
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

## Step 6 — Find caught and missed scams
A **caught scam** has actual label `1` and prediction `1`. A **missed scam** has actual label `1` but prediction `0`. The missed scams are central to our research question.

In [ ]:
# Cell 9: Add predictions and confidence scores to the external test table
external_results = external_test_data.copy()
external_results['prediction'] = external_predictions
external_results['scam_probability'] = model.predict_proba(X_external_tfidf)[:, 1]
external_results['result_type'] = np.select(
    [
        (external_results['label'] == 1) & (external_results['prediction'] == 1),
        (external_results['label'] == 1) & (external_results['prediction'] == 0),
        (external_results['label'] == 0) & (external_results['prediction'] == 0),
        (external_results['label'] == 0) & (external_results['prediction'] == 1),
    ],
    ['caught_scam', 'missed_scam', 'correct_legitimate', 'false_alarm'],
    default='unknown',
)

result_counts = external_results['result_type'].value_counts().rename_axis('result_type').to_frame('count')
result_counts

In [ ]:
# Cell 10: Display five missed scams for manual reading
missed_scams = external_results.loc[
    external_results['result_type'] == 'missed_scam',
    ['source', 'text', 'scam_probability'],
].sort_values('scam_probability').reset_index(drop=True)

caught_scams = external_results.loc[
    external_results['result_type'] == 'caught_scam',
    ['source', 'text', 'scam_probability'],
].sort_values('scam_probability', ascending=False).reset_index(drop=True)

print('Number of missed scams:', len(missed_scams))
display(missed_scams.head(5))

## Step 7 — Inspect influential words
Positive weights push the prediction toward scam. Negative weights push it toward legitimate. These weights describe the Logistic Regression model, but they do not prove why a person would trust an email.

In [ ]:
# Cell 11: Show the strongest learned word features
feature_names = np.array(vectorizer.get_feature_names_out())
weights = model.coef_[0]
top_scam = np.argsort(weights)[-15:][::-1]
top_legitimate = np.argsort(weights)[:15]

important_words = pd.DataFrame({
    'scam_word_or_phrase': feature_names[top_scam],
    'scam_weight': weights[top_scam],
    'legitimate_word_or_phrase': feature_names[top_legitimate],
    'legitimate_weight': weights[top_legitimate],
})
important_words

## Step 8 — Save results for Part 3 and the report

In [ ]:
# Cell 12: Save exact results and individual predictions
output_dir = Path('/content/hit401_part2_results')
output_dir.mkdir(exist_ok=True)
results.to_csv(output_dir / 'model_metrics.csv')
external_results.to_csv(output_dir / 'external_predictions.csv', index=False)
missed_scams.to_csv(output_dir / 'missed_scams.csv', index=False)
caught_scams.to_csv(output_dir / 'caught_scams.csv', index=False)
important_words.to_csv(output_dir / 'important_words.csv', index=False)

archive_path = Path('/content/hit401_part2_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_dir.glob('*.csv'):
        archive.write(file_path, arcname=file_path.name)

print('Saved:', archive_path)
print('Download this ZIP from the Colab Files panel.')

## What to tell the supervisor

> We trained a TF-IDF and Logistic Regression baseline using Nazario phishing emails and Enron legitimate emails. We first evaluated it on held-out emails from those familiar sources. We then performed a stronger external test using Nigerian Fraud and SpamAssassin, which were not used for training. We compared accuracy, precision, recall and F1-score, then exported the caught and missed scams for detailed analysis.

## Limitation
A performance difference between internal and external testing can be caused by changes in source, time period, writing style and label definition. It is evidence of limited transfer in this experiment, not proof that every scam detector will behave identically.